# Análise de Grafos - MOOC Actions

Definição formal, métricas, distribuição de graus P(k), ajuste por lei de potência, clustering e discussão sobre escala livre.

In [ ]:
import csv
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from algs4.digraph import Digraph

%matplotlib inline

## 1. Leitura do dataset

In [ ]:
print("Lendo dataset...")

node_map = {}
current_index = 0
edges = []

with open("act-mooc/mooc_actions.tsv", "r") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        user = "U_" + row["USERID"]
        target = "T_" + row["TARGETID"]

        if user not in node_map:
            node_map[user] = current_index
            current_index += 1

        if target not in node_map:
            node_map[target] = current_index
            current_index += 1

        edges.append((node_map[user], node_map[target]))

print("Construindo grafo...")

## 2. Construção do grafo e cálculo dos graus

In [ ]:
G = Digraph(current_index)
for u, v in edges:
    G.add_edge(u, v)

V = G.V
E = G.E

print("Calculando graus de entrada e saída...")

out_degrees = [len(list(G.adj[v])) for v in range(V)]
in_degrees = [0] * V
for v in range(V):
    for w in list(G.adj[v]):
        in_degrees[w] += 1

## 3. Definição formal e métricas básicas

In [ ]:
print("="*60)
print("DEFINIÇÃO FORMAL DO GRAFO")
print("="*60)
print("Tipo: dirigido (dígrafo)")
print("Ponderado: não (todas as arestas têm peso unitário)")
print("Temporal: não (análise estática; dados originais têm timestamp)")
print()
print("MÉTRICAS")
print("-"*40)
print(f"Ordem |V| = {V}")
print(f"Tamanho |E| = {E}")
densidade = E / (V * (V - 1)) if V > 1 else 0.0
print(f"Densidade = |E|/(|V|*(|V|-1)) = {densidade:.6e}")
grau_medio_entrada = sum(in_degrees) / V
grau_medio_saida = sum(out_degrees) / V
print(f"Grau médio (entrada) = {grau_medio_entrada:.4f}")
print(f"Grau médio (saída)   = {grau_medio_saida:.4f}")
print("="*60)

## 4. Distribuição P(k) e ajuste por lei de potência

In [ ]:
def calc_distribuicao(lista_graus):
    graus_filtrados = [k for k in lista_graus if k > 0]
    n = len(graus_filtrados)
    contagem = Counter(graus_filtrados)
    x = sorted(contagem.keys())
    y = [contagem[k] / n for k in x]
    return x, y

x_in, y_in = calc_distribuicao(in_degrees)
x_out, y_out = calc_distribuicao(out_degrees)

K_MIN = 2

def ajuste_lei_potencia(x, y, k_min=K_MIN):
    mask = [i for i in range(len(x)) if x[i] >= k_min]
    if len(mask) < 3:
        return None, None, None
    log_k = np.log(np.array([x[i] for i in mask], dtype=float))
    log_p = np.log(np.array([y[i] for i in mask], dtype=float))
    coeffs = np.polyfit(log_k, log_p, 1)
    gamma = -coeffs[0]
    pred = np.polyval(coeffs, log_k)
    ss_res = np.sum((log_p - pred) ** 2)
    ss_tot = np.sum((log_p - np.mean(log_p)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    return gamma, coeffs[1], r2

gamma_in, c_in, r2_in = ajuste_lei_potencia(x_in, y_in)
gamma_out, c_out, r2_out = ajuste_lei_potencia(x_out, y_out)

print("AJUSTE LEI DE POTÊNCIA P(k) ~ k^(-gamma)")
print("-"*50)
print("Método: regressão linear em log-log (MQO), k >= k_min =", K_MIN)
if gamma_in is not None:
    print(f"Grau de ENTRADA: gamma = {gamma_in:.3f}, R² = {r2_in:.4f}")
else:
    print("Grau de ENTRADA: insuficientes pontos para ajuste")
if gamma_out is not None:
    print(f"Grau de SAÍDA:   gamma = {gamma_out:.3f}, R² = {r2_out:.4f}")
else:
    print("Grau de SAÍDA:   insuficientes pontos para ajuste")

## 5. Coeficiente de clustering médio

In [ ]:
def clustering_medio_digrafo(G, V):
    total_c = 0.0
    count = 0
    adj = G.adj
    for v in range(V):
        vizinhos = list(adj[v])
        n = len(vizinhos)
        if n < 2:
            continue
        pares_conectados = 0
        for i in range(n):
            for j in range(i + 1, n):
                a, b = vizinhos[i], vizinhos[j]
                if b in adj[a] or a in adj[b]:
                    pares_conectados += 1
        c_v = pares_conectados / (n * (n - 1) / 2)
        total_c += c_v
        count += 1
    return total_c / count if count > 0 else 0.0

clustering_medio = clustering_medio_digrafo(G, V)
print("COEFICIENTE DE CLUSTERING MÉDIO")
print("-"*50)
print(f"Clustering médio (dígrafo, vizinhos de saída) = {clustering_medio:.6f}")

## 6. Gráficos: histogramas, P(k) linear e log-log

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(12, 14))
cor_entrada = 'dodgerblue'
cor_saida = 'tomato'
max_in = max(in_degrees)
max_out = max(out_degrees)

bins_in = range(0, max_in + 2, max(1, max_in // 50))
bins_out = range(0, max_out + 2, max(1, max_out // 50))

axs[0, 0].hist(in_degrees, bins=bins_in, color=cor_entrada, edgecolor='black', alpha=0.7)
axs[0, 0].set_title('Histograma - Grau de Entrada')
axs[0, 0].set_xlabel('Grau')
axs[0, 0].set_ylabel('Frequência (Nº de Vértices)')
axs[0, 0].set_yscale('log')
axs[0, 0].axvline(max_in, color='red', linestyle='--', linewidth=1)
axs[0, 0].text(max_in, 1.1, f"max={max_in}", rotation=90, va='bottom', ha='right', fontsize=8)
axs[0, 0].grid(True, axis='y', alpha=0.3)

axs[0, 1].hist(out_degrees, bins=bins_out, color=cor_saida, edgecolor='black', alpha=0.7)
axs[0, 1].set_title('Histograma - Grau de Saída')
axs[0, 1].set_xlabel('Grau')
axs[0, 1].set_ylabel('Frequência (Nº de Vértices)')
axs[0, 1].set_yscale('log')
axs[0, 1].axvline(max_out, color='red', linestyle='--', linewidth=1)
axs[0, 1].text(max_out, 1.1, f"max={max_out}", rotation=90, va='bottom', ha='right', fontsize=8)
axs[0, 1].grid(True, axis='y', alpha=0.3)

axs[1, 0].scatter(x_in, y_in, color=cor_entrada, alpha=0.6, edgecolors='none')
axs[1, 0].set_title('Distribuição Linear - Entrada')
axs[1, 0].set_xlabel('Grau (k)')
axs[1, 0].set_ylabel('P(k)')
axs[1, 0].grid(True, alpha=0.3)

axs[1, 1].scatter(x_out, y_out, color=cor_saida, alpha=0.6, edgecolors='none')
axs[1, 1].set_title('Distribuição Linear - Saída')
axs[1, 1].set_xlabel('Grau (k)')
axs[1, 1].set_ylabel('P(k)')
axs[1, 1].grid(True, alpha=0.3)

axs[2, 0].scatter(x_in, y_in, color=cor_entrada, alpha=0.6, edgecolors='none', label='P(k)')
if gamma_in is not None:
    k_fit = np.array([k for k in x_in if k >= K_MIN], dtype=float)
    p_fit = np.exp(c_in) * np.power(k_fit, -gamma_in)
    axs[2, 0].plot(k_fit, p_fit, 'k--', linewidth=1.5, label=f'ajuste ~ k^{{-{gamma_in:.2f}}}')
axs[2, 0].set_title('Distribuição Log-Log - Entrada')
axs[2, 0].set_xlabel('Grau (k)')
axs[2, 0].set_ylabel('P(k)')
axs[2, 0].set_xscale('log')
axs[2, 0].set_yscale('log')
axs[2, 0].legend(loc='upper right', fontsize=8)
axs[2, 0].grid(True, which="both", alpha=0.3)

axs[2, 1].scatter(x_out, y_out, color=cor_saida, alpha=0.6, edgecolors='none', label='P(k)')
if gamma_out is not None:
    k_fit = np.array([k for k in x_out if k >= K_MIN], dtype=float)
    p_fit = np.exp(c_out) * np.power(k_fit, -gamma_out)
    axs[2, 1].plot(k_fit, p_fit, 'k--', linewidth=1.5, label=f'ajuste ~ k^{{-{gamma_out:.2f}}}')
axs[2, 1].set_title('Distribuição Log-Log - Saída')
axs[2, 1].set_xlabel('Grau (k)')
axs[2, 1].set_ylabel('P(k)')
axs[2, 1].set_xscale('log')
axs[2, 1].set_yscale('log')
axs[2, 1].legend(loc='upper right', fontsize=8)
axs[2, 1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Discussão: a rede pode ser considerada de escala livre?

Uma rede é frequentemente considerada **de escala livre** quando:
1. A distribuição de graus P(k) segue aproximadamente uma **lei de potência** P(k) ~ k^(-γ).
2. O expoente **γ** costuma estar na faixa **2 < γ < 3** em muitas redes reais.
3. Há forte assimetria: poucos **hubs** (alto grau) e muitos nós de grau baixo.

**Interpretação:** Os valores de γ e R² do ajuste indicam em que medida P(k) se aproxima de uma lei de potência. Se γ ∈ (2, 3) e R² aceitável, a rede pode ser considerada de escala livre. O coeficiente de clustering complementa a análise.